# FraudIA Claims — Entrenamiento de Modelos ML
**Notebook Google Colab — sin necesidad de Google Drive**

## Pasos rápidos
1. Ejecuta la celda **0.1** → instala librerías
2. Ejecuta **0.2** → sube `claims_with_documents.csv` cuando lo pida
3. Ejecuta el resto en orden
4. Al final, la celda **6** descarga `fraudia_models.zip` a tu PC
5. Descomprime en la carpeta `models/` del proyecto

**Artefactos generados:**
```
fraud_model.pkl            → RandomForestClassifier
isolation_forest.pkl       → IsolationForest
scaler.pkl                 → StandardScaler
model_columns.json         → lista de features
metrics.json               → precision, recall, F1, AUC-ROC
shap_feature_importance.json
model_scores.csv           → scores ML para los 500 siniestros
```

## 0.1 Instalar dependencias

In [ ]:
!pip install shap xgboost -q
print('✓ Dependencias instaladas')

## 0.2 Subir el archivo de datos

Sube el archivo **`claims_with_documents.csv`** que está en `data/processed/` del proyecto.
*(Si tienes el proyecto en Drive, cambia `USE_DRIVE = True` y ajusta `DRIVE_PATH`)*

In [ ]:
# ─── CONFIGURACIÓN ────────────────────────────────────────────────────────────
USE_DRIVE  = False   # Cambiar a True si tienes el proyecto en Google Drive
DRIVE_PATH = '/content/drive/MyDrive/hackiathon-aseguradora-del-sur'
# ──────────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import json, joblib, warnings, zipfile
from pathlib import Path

MODELS_DIR = Path('/content/fraudia_models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = Path(DRIVE_PATH) / 'data' / 'processed' / 'claims_with_documents.csv'
    df = pd.read_csv(DATA_PATH)
    print(f'✓ Datos cargados desde Drive: {df.shape}')
else:
    from google.colab import files
    print('Selecciona el archivo claims_with_documents.csv:')
    uploaded = files.upload()          # abre el diálogo de carga
    fname = list(uploaded.keys())[0]
    import io
    df = pd.read_csv(io.BytesIO(uploaded[fname]))
    print(f'✓ Datos cargados: {df.shape[0]} siniestros, {df.shape[1]} columnas')

df.head(3)

## 0.3 Importaciones

In [ ]:
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, roc_auc_score,
    precision_score, recall_score, f1_score
)
import shap
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
print('✓ Importaciones OK')

## 1. Preparación de features

In [ ]:
FEATURE_COLS = [
    'monto_reclamado', 'monto_estimado',
    'dias_desde_inicio_poliza', 'dias_hasta_fin_poliza',
    'dias_ocurrencia_reporte', 'historial_siniestros_asegurado',
    'similitud_narrativa', 'ratio_monto_suma', 'cantidad_documentos',
    'n_reclamos_12_meses', 'n_reclamos_historico', 'reclamos_rc_sin_tercero',
    'antiguedad_anios', 'n_siniestros_proveedor', 'promedio_monto_proveedor',
    'doc_factura_alterada', 'doc_ruc_invalido', 'doc_parte_tardio',
    'doc_sin_denuncia_previa', 'doc_sin_testigos', 'doc_robo', 'doc_perdida_total',
    'proveedor_lista_restrictiva', 'alerta_borde_inicio', 'alerta_borde_fin',
    'reporte_tardio', 'narrativa_similar', 'narrativa_clonada',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
print(f'✓ Features disponibles: {len(FEATURE_COLS)}')
print(FEATURE_COLS)

In [ ]:
X = df[FEATURE_COLS].copy()

for col in X.select_dtypes(include='bool').columns:
    X[col] = X[col].astype(int)

X = X.fillna(X.median(numeric_only=True))

print(f'X shape: {X.shape}  |  nulos: {X.isnull().sum().sum()}')

In [ ]:
def create_label(row):
    if row.get('doc_factura_alterada', False):        return 1
    if row.get('doc_ruc_invalido', False):            return 1
    if row.get('proveedor_lista_restrictiva', False): return 1
    sim  = row.get('similitud_narrativa', 0) or 0
    dias = row.get('dias_desde_inicio_poliza', 999) or 999
    if sim >= 0.85 and dias <= 30:                    return 1
    if row.get('narrativa_clonada', False):           return 1
    if row.get('doc_sin_denuncia_previa', False) and row.get('doc_robo', False): return 1
    return 0

y = df.apply(create_label, axis=1)
print(f'Distribución de etiquetas:')
print(y.value_counts().to_string())
print(f'\nProporción sospechosos: {y.mean():.1%}')

## 2. Isolation Forest

In [ ]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

isof = IsolationForest(n_estimators=200, contamination=0.15, random_state=42, n_jobs=-1)
isof.fit(X_scaled)

raw   = isof.score_samples(X_scaled)
s_min, s_max = raw.min(), raw.max()
isof_scores = ((s_min - raw) / (s_min - s_max) * 100).clip(0, 100)
df['score_isolation_forest'] = isof_scores.round(1)

print(f'✓ Isolation Forest entrenado')
print(f'  Score medio: {isof_scores.mean():.1f}  |  Anómalos (>50): {(isof_scores > 50).sum()}')

## 3. Random Forest

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=3,
    class_weight='balanced', random_state=42, n_jobs=-1,
)
rf.fit(X_train, y_train)

y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print('✓ Random Forest entrenado')
print(classification_report(y_test, y_pred, target_names=['Legítimo', 'Sospechoso']))

In [ ]:
metrics = {
    'precision':    round(float(precision_score(y_test, y_pred)),  3),
    'recall':       round(float(recall_score(y_test, y_pred)),     3),
    'f1':           round(float(f1_score(y_test, y_pred)),         3),
    'auc_roc':      round(float(roc_auc_score(y_test, y_proba)),   3),
    'cv_f1_mean':   round(float(cross_val_score(rf, X, y, cv=5, scoring='f1').mean()), 3),
    'n_train':      int(len(X_train)),
    'n_test':       int(len(X_test)),
    'n_features':   int(len(FEATURE_COLS)),
    'feature_cols': FEATURE_COLS,
}

rf_scores_all = rf.predict_proba(X)[:, 1] * 100
df['score_random_forest'] = rf_scores_all.round(1)

print(json.dumps({k: v for k, v in metrics.items() if k != 'feature_cols'}, indent=2))

## 4. SHAP — Explicabilidad

In [ ]:
explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

sv = shap_values[1] if isinstance(shap_values, list) else shap_values

shap_importance = dict(sorted(
    zip(FEATURE_COLS, np.abs(sv).mean(axis=0).tolist()),
    key=lambda x: x[1], reverse=True
))
shap_importance = {k: round(v, 6) for k, v in shap_importance.items()}

print('Top 10 features por importancia SHAP:')
for feat, val in list(shap_importance.items())[:10]:
    print(f'  {feat:<40s}: {val:.4f}')

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(sv, X_test, feature_names=FEATURE_COLS,
                  plot_type='bar', show=False, max_display=15)
plt.title('Importancia SHAP — FraudIA Claims')
plt.tight_layout()
plt.savefig(str(MODELS_DIR / 'shap_summary.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Guardar artefactos

In [ ]:
joblib.dump(rf,     MODELS_DIR / 'fraud_model.pkl')
joblib.dump(isof,   MODELS_DIR / 'isolation_forest.pkl')
joblib.dump(scaler, MODELS_DIR / 'scaler.pkl')

(MODELS_DIR / 'model_columns.json').write_text(
    json.dumps(FEATURE_COLS, ensure_ascii=False, indent=2))
(MODELS_DIR / 'metrics.json').write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2))
(MODELS_DIR / 'shap_feature_importance.json').write_text(
    json.dumps(shap_importance, ensure_ascii=False, indent=2))

# CSV con scores del modelo para los 500 siniestros
df[['id_siniestro', 'score_isolation_forest', 'score_random_forest']].to_csv(
    MODELS_DIR / 'model_scores.csv', index=False, encoding='utf-8-sig')

print('Artefactos en /content/fraudia_models/:')
for f in sorted(MODELS_DIR.glob('*')):
    print(f'  {f.name}: {f.stat().st_size / 1024:.1f} KB')

## 6. Verificación y descarga

Esta celda verifica que los artefactos cargan correctamente y descarga
**`fraudia_models.zip`** a tu PC. Luego extrae su contenido en la carpeta `models/` del proyecto.

In [ ]:
# Verificación
rf_ok     = joblib.load(MODELS_DIR / 'fraud_model.pkl')
scaler_ok = joblib.load(MODELS_DIR / 'scaler.pkl')

sin0005 = X[df['id_siniestro'] == 'SIN-0005']
if len(sin0005) > 0:
    prob = rf_ok.predict_proba(sin0005)[0, 1]
    print(f'SIN-0005 (caso más sospechoso) → probabilidad RF: {prob:.1%}')

print(f'\n=== Resumen ===')
print(f'  Precision : {metrics["precision"]}')
print(f'  Recall    : {metrics["recall"]}')
print(f'  F1        : {metrics["f1"]}')
print(f'  AUC-ROC   : {metrics["auc_roc"]}')
print(f'  CV F1 mean: {metrics["cv_f1_mean"]}')

In [ ]:
# Empaquetar todo en un ZIP y descargar
zip_path = Path('/content/fraudia_models.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in MODELS_DIR.glob('*'):
        zf.write(f, f.name)   # guarda sin subdirectorio

print(f'ZIP creado: {zip_path.stat().st_size / 1024:.0f} KB')
print('Descargando...')

from google.colab import files
files.download(str(zip_path))

print()
print('Pasos finales:')
print('  1. Espera a que se descargue fraudia_models.zip')
print('  2. Extrae su contenido en la carpeta models/ del proyecto')
print('  3. El dashboard usará automáticamente los modelos entrenados')